In [16]:
import json
import random
import os
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer


In [17]:
# Performs reservoir sampling on json stream
# @param filePath - Path to source json file
# @param sampleSize - Number of records to retain
# @returns - List of sampled dictionaries
def sampleStream(filePath, sampleSize):
    sampledRecords = []
    validCount = 0
    with open(filePath, "r", encoding="utf-8") as fileDescriptor:
        for line in fileDescriptor:
            try:
                record = json.loads(line)
            except json.JSONDecodeError:
                continue
            if "venue" not in record or "abstract" not in record:
                continue
            yearValue = record.get("year")
            try:
                yearValue = float(yearValue)
            except (TypeError, ValueError):
                yearValue = float("nan")
            authorsList =[]
            for author in record.get("authors",[]):
                if isinstance(author, dict) and "name" in author:
                    authorsList.append(author["name"])
            referenceList = record.get("references",[])
            if not isinstance(referenceList, list):
                referenceList =[]
            processedRecord = {
                "id": record.get("id", ""),
                "venue": str(record.get("venue", "")),
                "year": yearValue,
                "n_citation": int(record.get("n_citation", 0)),
                "abstract": str(record.get("abstract", "")),
                "authors": authorsList,
                "references": referenceList
            }
            if validCount < sampleSize:
                sampledRecords.append(processedRecord)
            else:
                replacementIndex = random.randint(0, validCount)
                if replacementIndex < sampleSize:
                    sampledRecords[replacementIndex] = processedRecord
            validCount += 1
    return sampledRecords

random.seed(32)
rawData = sampleStream("../data/dblp-ref/dblp-ref-0.json", 50000)
baseDataFrame = pd.DataFrame(rawData)


In [18]:
# Fills missing years using median year of venue
# @param dataFrame - Dataset containing year and venue columns
# @returns - Series containing imputed years
def imputeMissingYears(dataFrame):
    imputedYears = dataFrame.groupby("venue")["year"].transform(lambda group: group.fillna(group.median()))
    return imputedYears.fillna(imputedYears.median())

baseDataFrame["year"] = imputeMissingYears(baseDataFrame)


In [19]:
# Generates numerical features from text
# @param textSeries - Text data array
# @param maxWords - Maximum features to retain
# @returns - Numerical feature matrix
def extractTextFeatures(textSeries, maxWords):
    vectorizer = TfidfVectorizer(max_features=maxWords, stop_words="english")
    featureMatrix = vectorizer.fit_transform(textSeries)
    featureNames = vectorizer.get_feature_names_out()
    return pd.DataFrame(featureMatrix.toarray(), columns=featureNames)

textFeatures = extractTextFeatures(baseDataFrame["abstract"], 50)


In [20]:
finalDataFrame = pd.concat([baseDataFrame.drop(columns=["abstract"]), textFeatures], axis=1)
os.makedirs("../../data", exist_ok=True)
finalDataFrame.to_parquet("../data/processed_dblp.parquet")
